In [0]:
CREATE TABLE IF NOT EXISTS employee_payments (
emp_id INT PRIMARY KEY,
emp_name VARCHAR(50),
department VARCHAR(30),
base_salary DECIMAL(10,2),
bonus DECIMAL(10,2),
joining_date DATE
);


INSERT INTO employee_payments VALUES
(1,'karthik','Data',75000.75,5000.50,'2019-03-15'),
(2,'veena','HR',65000.40,4000.25,'2021-06-20'),
(3,'ravi','Data',85000.90,6000.75,'2016-01-10'),
(4,'anil','Finance',70000.10,NULL,'2020-09-01'),
(5,'suresh','HR',60000.55,3000.30,'2022-11-25');

num_affected_rows,num_inserted_rows
5,5


### For each employee:
· Convert emp_name to proper case ---upper /lower ---Initcap (CamelCase)

· Calculate total income = base_salary + bonus (NULL safe) +

· Round total income to nearest integer

· Extract joining year


In [0]:
--· Convert emp_name to proper case ---upper /lower ---Initcap (CamelCase)
select initcap(emp_name), sum(base_salary+coalesce(bonus,0)) as total_income, round(sum(base_salary+coalesce(bonus,0))) as rounded_income, year(joining_date)
from employee_payments
group by emp_name, joining_date

initcap(emp_name),total_income,rounded_income,year(joining_date)
Karthik,80001.25,80001,2019
Veena,69000.65,69001,2021
Ravi,91001.65,91002,2016
Anil,70000.10,70000,2020
Suresh,63000.85,63001,2022


· Use CASE to classify:

o Senior if experience > 7 years

o Mid if between 4 and 7

o Junior otherwise


In [0]:
ALTER TABLE employee_payments
ADD COLUMN joining_year INT;
UPDATE employee_payments
SET joining_year = EXTRACT(YEAR FROM joining_date);

num_affected_rows
5


In [0]:
SELECT *,(YEAR(CURRENT_DATE) - joining_year) AS experience,
CASE
    WHEN experience > 7 THEN 'Senior'
    WHEN experience BETWEEN 4 AND 7 THEN 'Mid'
    ELSE 'Junior'
END AS experience_level

FROM employee_payments;

emp_id,emp_name,department,base_salary,bonus,joining_date,joining_year,experience,experience_level
1,karthik,Data,75000.75,5000.50,2019-03-15,2019,7,Mid
2,veena,HR,65000.40,4000.25,2021-06-20,2021,5,Mid
3,ravi,Data,85000.90,6000.75,2016-01-10,2016,10,Senior
4,anil,Finance,70000.10,null,2020-09-01,2020,6,Mid
5,suresh,HR,60000.55,3000.30,2022-11-25,2022,4,Mid


### Question 2


In [0]:
CREATE TABLE IF NOT EXISTS orders_delivery (
order_id INT,
customer_name VARCHAR(50),
order_date DATE,
delivery_date DATE,
order_amount DECIMAL(10,2)
);
-- Insert Data
INSERT INTO orders_delivery VALUES
(101,'rajesh','2025-01-01','2025-01-05',12500.75),
(102,'meena','2025-01-10','2025-01-10',8400.40),
(103,'arun','2025-01-15','2025-01-20',15600.90),
(104,'pooja','2025-01-18',NULL,9200.10);

num_affected_rows,num_inserted_rows
4,4


For each order:

· Uppercase customer name

· Calculate delivery days using date difference

· Replace NULL delivery date with today

· Truncate order amount to 1 decimal

· Use CASE:

o Same-day

o Delayed (>3 days)

o Pending


In [0]:
SELECT 
    order_id,
    UPPER(customer_name) AS customer_name,
    order_date,
    -- Replace NULL with today
    COALESCE(delivery_date, CURRENT_DATE) AS delivery_date,
    -- Delivery days
    DATEDIFF(COALESCE(delivery_date, CURRENT_DATE), order_date) AS delivery_days,
    -- Truncate amount not available so using round
    ROUND(order_amount, 1) AS order_amount,
    CASE 
        WHEN delivery_date IS NULL THEN 'Pending'
        WHEN DATEDIFF(delivery_date, order_date) = 0 THEN 'Same-day'
        WHEN DATEDIFF(delivery_date, order_date) > 3 THEN 'Delayed'
        ELSE 'On-time'
    END AS delivery_status
FROM orders_delivery;

order_id,customer_name,order_date,delivery_date,delivery_days,order_amount,delivery_status
101,RAJESH,2025-01-01,2025-01-05,4,12500.8,Delayed
102,MEENA,2025-01-10,2025-01-10,0,8400.4,Same-day
103,ARUN,2025-01-15,2025-01-20,5,15600.9,Delayed
104,POOJA,2025-01-18,2026-04-10,447,9200.1,Pending


###QUESTION 3: Customer Spending Pattern

In [0]:
CREATE TABLE customer_spending (
cust_id INT,
cust_name VARCHAR(50),
city VARCHAR(30),
purchase_amount DECIMAL(10,2),
purchase_date DATE
);
--Insert Data
INSERT INTO customer_spending VALUES
(1,'amit','mumbai',12000.75,'2024-12-01'),
(2,'neha','delhi',8500.40,'2024-12-15'),
(3,'rohit','mumbai',15500.90,'2024-11-20'),
(4,'kavya','chennai',6000.10,'2024-10-05');

num_affected_rows,num_inserted_rows
4,4


Display:
· Customer name with first letter capitalized
· Month name of purchase
· Rounded purchase amount
· Absolute value of purchase (defensive logic)
· CASE:
o High spender > 15000
o Medium 8000–15000


In [0]:
SELECT 
    cust_id,
    INITCAP(cust_name) AS cust_name,
    MONTHNAME(purchase_date) AS purchase_month,
    ROUND(purchase_amount) AS rounded_amount,
    ABS(purchase_amount) AS absolute_amount,
    CASE 
        WHEN purchase_amount > 15000 THEN 'High'
        WHEN purchase_amount BETWEEN 8000 AND 15000 THEN 'Medium'
        ELSE 'Low'
    END AS spending_category
FROM customer_spending;

cust_id,cust_name,purchase_month,rounded_amount,absolute_amount,spending_category
1,Amit,Dec,12001,12000.75,Medium
2,Neha,Dec,8500,8500.40,Medium
3,Rohit,Nov,15501,15500.90,High
4,Kavya,Oct,6000,6000.10,Low


###QUESTION 4: Subscription Validity Check

In [0]:
CREATE TABLE subscriptions (
user_id INT,
user_email VARCHAR(100),
start_date DATE,
end_date DATE,
subscription_fee DECIMAL(10,2)
);
--Insert Data
INSERT INTO subscriptions VALUES
(1,'karthik@gmail.com','2024-01-01','2025-01-01',12000.50),
(2,'veena@yahoo.com','2024-06-15','2024-12-15',8500.75),
(3,'ravi@hotmail.com','2023-03-01','2024-03-01',15000.90);


num_affected_rows,num_inserted_rows
3,3


For each user:

· Extract email domain

· Calculate subscription duration in months

· Format fee with commas

· Find remaining days from today

· CASE:

o Active

o Expiring Soon (≤30 days)

o Expired


In [0]:
SELECT 
    user_id,
    -- Extract domain
    SUBSTRING_INDEX(user_email, '@', -1) AS domain,
    -- Duration in months
    MONTHS_BETWEEN(end_date, start_date) AS duration_months,
    -- Format fee
    FORMAT_NUMBER(subscription_fee, 2) AS formatted_fee,
    -- Remaining days
    DATEDIFF(end_date, CURRENT_DATE) AS remaining_days,
    CASE 
        WHEN end_date < CURRENT_DATE THEN 'Expired'
        WHEN DATEDIFF(end_date, CURRENT_DATE) <= 30 THEN 'Expiring Soon'
        ELSE 'Active'
    END AS status
FROM subscriptions;

user_id,domain,duration_months,formatted_fee,remaining_days,status
1,gmail.com,12.0,"12,000.50",-464,Expired
2,yahoo.com,6.0,"8,500.75",-481,Expired
3,hotmail.com,12.0,"15,000.90",-770,Expired


###QUESTION 5: Loan EMI Risk Categorization

In [0]:
CREATE TABLE loan_details (
loan_id INT,
customer_name VARCHAR(50),
loan_amount DECIMAL(12,2),
interest_rate DECIMAL(5,2),
loan_start DATE
);
-- Insert Data
INSERT INTO loan_details VALUES
(201,'suresh',500000.75,8.5,'2022-01-10'),
(202,'mahesh',750000.40,9.2,'2021-05-20'),
(203,'anita',300000.90,7.8,'2023-07-01');

num_affected_rows,num_inserted_rows
3,3


Compute:
· Monthly interest using power function
· Years since loan start
· Round EMI
· Uppercase customer name
· CASE:
o High Risk if interest > 9
o Medium Risk
o Low Risk

In [0]:
SELECT 
    loan_id,
    UPPER(customer_name) AS customer_name,
    -- Years since loan start
    YEAR(CURRENT_DATE) - YEAR(loan_start) AS years,
    -- Monthly interest using POWER
    POWER(1 + interest_rate/100, 1/12) AS monthly_interest,
    -- EMI (simplified)
    ROUND(loan_amount * (interest_rate/100)) AS emi,
    CASE 
        WHEN interest_rate > 9 THEN 'High Risk'
        WHEN interest_rate BETWEEN 8 AND 9 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS risk_level
FROM loan_details;

loan_id,customer_name,years,monthly_interest,emi,risk_level
201,SURESH,4,1.0068214933659623,42500,Medium Risk
202,MAHESH,5,1.0073612011869553,69000,High Risk
203,ANITA,3,1.0062785842352273,23400,Low Risk


###QUESTION 6: Employee Attendance Evaluation

In [0]:
CREATE TABLE IF NOT EXISTS attendance (
emp_id INT,
emp_name VARCHAR(50),
total_days INT,
present_days INT,
record_date DATE
);
-- Insert Data
INSERT INTO attendance VALUES
(1,'karthik',30,28,'2025-01-31'),
(2,'veena',30,22,'2025-01-31'),
(3,'ravi',30,18,'2025-01-31');

num_affected_rows,num_inserted_rows
3,3


In [0]:
select lower(emp_name) ,round(present_days/total_days*100),

monthname(record_date),present_days-total_days,
case when round(present_days/total_days*100)>=90 then "Excellent"
when round(present_days/total_days*100) between 75 and 89 then "Average"
else 'Poor'
end
from attendance

lower(emp_name),round(present_days/total_days*100),monthname(record_date),present_days-total_days,"CASEWHENround(present_days/total_days*100)>=90THEN""Excellent""WHENround(present_days/total_days*100)BETWEEN75AND89THEN""Average""ELSE'Poor'END"
karthik,93.0,Jan,-2,Excellent
veena,73.0,Jan,-8,Poor
ravi,60.0,Jan,-12,Poor
